# Phase VII — confirmatory linear probes (resume-only)

This notebook resumes **only** the confirmatory artist-disjoint linear-probe stage from the already-computed full ArtBench-10 feature matrix in Google Drive. It does **not** re-download ArtBench, re-extract B90/G44, or recompute ordinal features.

Checkpoints are written after every outer fold, so the run can safely resume after a Colab interruption.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess, sys

BRANCH = 'multiscale-corpus-analysis'
REPO_URL = 'https://github.com/ardominguezm/painting-geometry.git'
REPO_DIR = Path('/content/painting-geometry')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)], check=True)
commit = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], text=True).strip()
print('Repository commit:', commit)
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','pandas','scikit-learn'], check=True)


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/painting_geometry_phase7_full')
FEATURES = DRIVE_ROOT / 'results' / 'artbench_full_features_with_ordinal.csv'
OUT = DRIVE_ROOT / 'results' / 'phase7_confirmatory_linear_probes'
LOG = DRIVE_ROOT / 'phase7_confirmatory_resume.log'

assert FEATURES.exists(), f'Missing full feature matrix: {FEATURES}'
print('Input:', FEATURES)
print('Output:', OUT)
print('Log:', LOG)

cmd = [sys.executable, '-u', str(REPO_DIR/'scripts'/'run_phase7_full_confirmatory.py'),
       '--features', str(FEATURES), '--output-dir', str(OUT),
       '--outer-folds', '5', '--inner-folds', '3', '--n-boot', '5000']

with open(LOG, 'a', buffering=1) as log:
    log.write('\n\n=== NEW RESUME RUN ===\n')
    log.write('commit=' + commit + '\n')
    proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
    rc = proc.wait()

if rc != 0:
    raise RuntimeError(f'Confirmatory resume stopped with exit code {rc}. Checkpoints in {OUT} are preserved; rerun this cell to continue.')
print('Confirmatory Phase VII complete ✓')


In [ ]:
import zipfile

LIGHT = DRIVE_ROOT / 'painting_geometry_phase7_full_results_LIGHT.zip'
RESULTS = DRIVE_ROOT / 'results'
EXCLUDE_BIG = {'artbench_full_B90_G44_features.csv', 'artbench_full_features_with_ordinal.csv',
               'artbench_full_features_with_ordinal.ordinal_checkpoint.csv'}

with zipfile.ZipFile(LIGHT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS.rglob('*'):
        if not p.is_file() or p.name in EXCLUDE_BIG or '_representation_checkpoints' in p.parts:
            continue
        z.write(p, p.relative_to(RESULTS))

print('LIGHT results ZIP:', LIGHT)
print(f'Size: {LIGHT.stat().st_size/1e6:.1f} MB')

try:
    from google.colab import files
    files.download(str(LIGHT))
except Exception as exc:
    print('Automatic browser download skipped:', exc)
